# Fig. 3: Effective model along a cut in pump rate

Effective lasing model (cooling atoms and mode $b$ adiabatically eliminated: the squeezed mode $a_s$ decays at rate $\kappa_f$ and is shifted by $\Delta$) for $N_c = 3000$, $N_g = 400$ and $r = 0.5$, as a function of the pump rate $w^{\rm p}$ (lines in **panels b, c**).

**Outputs** (`../data/P_Ng/`): `eff_n.npy`, `eff_linewidth.npy`, `eff_SdagS.npy`, `eff_pop_gain.npy`.

In [ ]:
using QuantumCumulants
using ModelingToolkit
using OrdinaryDiffEq
using WignerSymbols
using LinearAlgebra
using ProgressMeter
using NPZ

In [ ]:
Threads.nthreads()

In [ ]:
function dipole_me(F,Fp,m,p) # Dipole matrix element of the D1 line between |F,m> and |F',m+p> (Rb87, I = 3/2)
    
    J_g = 1/2
    J_e = 1/2
    I = 3/2

    sixj = Float64.(wigner6j(J_g, J_e, 1, Fp, F, I))
    cg = Float64.(clebschgordan(Fp, m+p, 1, -p, F, m))

    return (-1)^(Fp+J_g+1+I)*sqrt((2*Fp+1)*(2*J_g+1))*cg*sixj
end

## Parameters

In [ ]:
# Grid
num_O = 150
num_Nl = 1

# Rb87 level structure (B in G; angular frequencies, with the frequency in GHz)
B = 2.7
gamma = 2*pi*5.75*1e-3 # D1 linewidth
gamma_D2 = 2*pi*6.06*1e-3 # D2 linewidth
w_t = 2*pi*377107.463380
w_t_D2 = 2*pi*384230.48446
w_e = 2*pi*(6.834-3*0.7*1e-3*B)
w_s = w_t+2*pi*0.30544+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*2*0.23*1e-3*B
w_r = w_t-2*pi*0.50906+2*pi*4.271677-2*pi*0.7*1e-3*B+2*pi*0.23*1e-3*B
w_u = w_t_D2-2*pi*0.22985+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*0.93*1e-3*B

# Number of cooling (N_atom_I = N_c) and gain (N_atom_II = N_g) atoms
N_atom_I = 3000
N_atom_II = [400]
N_atom = N_atom_I .+ N_atom_II'

# Auxiliary cavity mode b
gb = 3*2*pi*1e-3
omega = 40*gamma_D2
w_b = w_u + 2*2pi
kappa_b = 20*2*pi*1e-3

# Main cavity mode a
kappa = 0.07*2*pi*1e-3
gr = 0.78*2*pi*1e-3
gs = gr*abs(dipole_me(2,2,-2,0)/dipole_me(1,1,-1,0))
r = 0.5 # Squeezing parameter
eps_cav = 1*1e-3*2*pi
w_cav = (1/3)*(w_e+4*w_r-w_s)+eps_cav

## Adiabatic elimination

Laser frequencies, rotating-frame detunings and effective couplings obtained after eliminating the excited levels $u$, $r$, $s$ and then the modes $b$ and $s$ (steps 1–7), and the effective pumping, decay and dephasing rates of the atoms (step 8).

In [ ]:
# Step 1: find convenient frequencies for the lasers

w_lr = (-w_cav+3*w_r-w_s).-sqrt.(4*gr^2*N_atom.+(w_cav-w_r)^2)
w_ls = -2*w_lr.+(4*w_r-w_s)
w_l = 0.5*(w_b.+w_lr.-w_ls.+w_u.+sqrt.(4*gb^2*N_atom_I.+(w_b-w_u)^2 .-omega^2))

# Step 2: move to rotating frame

delta_b = w_b .- w_l .- (w_ls.-w_lr)./2
delta_u = w_u .- w_l .- (w_ls.-w_lr)./2
delta_1 = w_e.-(w_ls.-w_lr)./2
delta_s = w_s.-w_ls
delta_r = w_r.-(w_ls.+w_lr)./2
delta_cav = w_cav.-(w_ls.+w_lr)./2

# Step 3: eliminate level u

w_0_b = -0.25.*(omega.^2)./delta_u
w_c_b = delta_b.-(N_atom_I*gb^2)./delta_u
chi_b = (gb^2)./delta_u
g = -omega.*gb./(2*delta_u)

# Step 4: find Rabi frequencies

omega_r = sqrt.((w_ls.-2*w_e.-w_lr)./(1 ./(w_lr.+w_ls.-2*w_r).-2*tanh.(r).^2 ./(w_ls.-w_s)))
omega_s = (gr/gs)*omega_r.*tanh.(r).*(delta_s./delta_r)

# Step 5: eliminate levels r and s

w_0 = delta_1.-0.25.*((omega_r.^2)./delta_r.-(omega_s.^2)./delta_s)
w_c = delta_cav.-(N_atom*gr^2)./delta_r
chi = (gr^2)./delta_r.-(gs^2)./delta_s
lambda_r = -omega_r.*gr./(2*delta_r)
lambda_s = -omega_s.*gs./(2*delta_s)
g_02 = (lambda_r.^2).-lambda_s.^2

alfa = omega_r./(2*delta_r)
beta = gr./delta_r
delta = omega_s./(2*delta_s)
xi = gs./delta_s
alfa_b = omega./(2*delta_u)
beta_b = gb./delta_u

dec_eff = gamma*dipole_me(1,1,-1,0)^2*alfa.^2
P_eff = gamma*dipole_me(2,2,-2,0)^2*delta.^2
dep_eff = (gamma*dipole_me(1,2,-1,-1)^2*delta.^2 .+ gamma*dipole_me(2,1,-2,+1)^2*alfa.^2)/4
dec_eff_b = gamma_D2*(5/12)*alfa_b.^2
dep_eff_b = gamma_D2*(1/10)*alfa_b.^2/4

# Step 6: Eliminate mode b

dec = (g.^2*kappa_b)./((w_c_b.-w_0_b).^2 .+ kappa_b^2/4)
delta_lamb = w_0_b .+ w_0 .+ N_atom_I.*(g.^2).*(w_c_b.-w_0_b)./((kappa_b^2/4).+(w_c_b.-w_0_b).^2) # coefficient of s's term

# Step 7: Eliminate mode s

dec_f = dec.*g_02./(dec.^2/4 .+ (delta_lamb./N_atom_I).^2)
delta_lamb_f = (g_02.*delta_lamb.*N_atom_I)./((delta_lamb.^2).+(N_atom_I.*dec/2).^2)

# Step 8: include pumping and dephasing

dec_T = dec_eff
dep_T = dep_eff

P = logrange(1e-6,1.798e-2,num_O)
P_T = P.+P_eff

C = (2*N_atom_II'.*g_02)./(((P_T.+dec_T)/2 .+ 2*dep_T).*(dec_f.+kappa))./(1 .+(4*delta_lamb_f.^2)./((kappa.+dec_f).+(P_T.+dec_T.+4*dep_T)).^2)
n_q = (P_T.-dec_T)./(2*(kappa.+dec_f)).*N_atom_II'
control = P_T./(kappa.+dec_f).-((P_T.+dec_T)./(2*(kappa.+dec_f))).*(1 .+ 1 ./ C)

purc = 4*g_02./(dec_f.+kappa)

## Cumulant expansion of the effective model

In [ ]:
@rnumbers Δ_c λ_r λ_s κ κ_f R N p γ_ϕ γ Λ # Parameters

h_cav = FockSpace(:cavity) # Hilbert space of the cavity
h_atom = NLevelSpace(:atom,(:g,:e)) # Hilbert space of the atom
h = h_cav⊗h_atom # Complete Hilbert space

as = Destroy(h,:as)
a = as*cosh(R)-as'*sinh(R)
σ(x,y,k) = IndexedOperator(Transition(h, :σ, x, y), k) # Operators

i = Index(h, :i, N, h_atom)
j = Index(h, :j, N, h_atom)

H = -Δ_c*as'*as + ∑(λ_r*(a*σ(:e,:g,i)+a'*σ(:g,:e,i)) + λ_s*(a'*σ(:e,:g,i)+a*σ(:g,:e,i)),i)

D = [a;as;σ(:e,:g,i);σ(:e,:e,i)-σ(:g,:g,i);σ(:g,:e,i)] # Collapse operators
rates = [κ; κ_f; p; γ_ϕ; γ]

In [ ]:
ops = [as] # Operators whose equations I want to derive
eqs = meanfield(ops,H,D;rates=rates,order=2) # Derive set of equations and factorize correlators for closing the set of equations
me_comp = complete!(eqs) # Automatically complete the system
eqs_sc = scale(me_comp) # Rescales the equations using symmetries

@named sys = System(eqs_sc)

In [ ]:
param_list = (Δ_c, λ_r, λ_s, κ, κ_f, R, N, p, γ_ϕ, γ, Λ)

## Steady state

Each point is integrated repeatedly (up to 10 times) until $\langle a_s^\dagger a_s\rangle$ and $\langle a_s^2\rangle$ stop changing over the integration window, re-using the previous solution as initial condition; observables are then time-averaged over one more window. Points that do not converge (`iterations > 10`) are non-stationary.

In [ ]:
n = zeros(num_O,num_Nl) # Photon number of the squeezed mode a_s
M = zeros(num_O,num_Nl) # Excited-state population of one gain atom
corr = zeros(num_O,num_Nl) # <S⁺S> of the gain atoms

it = zeros(num_O,num_Nl)
dist = zeros(num_O,num_Nl)
dist2 = zeros(num_O,num_Nl)

progress = Progress(num_O*num_Nl)
safe_lock = ReentrantLock()

Threads.@threads for l in 1:num_Nl

    u0 = zeros(ComplexF64, length(eqs_sc))
    sol = nothing
    p0 = nothing
    dict = nothing
    prob = nothing

    for c in 1:num_O

        dist[c,l] = 10
        dist2[c,l] = 1

        while (dist[c,l]>1 || dist2[c,l]>0.01) && it[c,l] < 10

            p0 = (delta_lamb_f[l], lambda_r[l], lambda_s[l], kappa, dec_f[l], r, N_atom_II[l], P_T[c,l], dep_T[l], dec_T[l], 0)
            dict = merge(Dict(unknowns(sys) .=> u0), Dict(param_list .=> p0))

            prob = ODEProblem(sys,dict,(0.0,30000.0))
            sol = solve(prob, Tsit5(); abstol=1e-8, reltol=1e-8, maxiters=1e6)

            lock(safe_lock) do
                dist_new = maximum(abs.(sol[as'as]))-minimum(abs.(sol[as'as]))
                dist2[c,l] = maximum(abs.(sol[as*as]))-minimum(abs.(sol[as*as]))

                if abs(dist_new - dist[c,l])/dist[c,l]<0.02 && dist[c,l] > 10
                    dist[c,l] = dist_new
                    it[c,l] = 10
                else
                    dist[c,l] = dist_new
                end
            end

            u0 = sol.u[end]

            it[c,l] = it[c,l]+1

            if sol.retcode != ReturnCode.Success
                dist[c,l] = 0.0
                break
            end

        end

        if sol.retcode != ReturnCode.Success

            n[c,l] = NaN
            M[c,l] = NaN
            corr[c,l] = NaN

        else
            prob = ODEProblem(sys,dict,(0.0,30000.0))
            sol = solve(prob, Tsit5(); abstol=1e-8, reltol=1e-8, maxiters=1e6)

            u0 = sol.u[end]

            n_t = length(sol.t)

            lock(safe_lock) do
                n[c,l] = sum(real.(sol[as'*as]))/n_t
                M[c,l] = sum(real.(sol[σ(:e,:e,1)]))/n_t
                corr[c,l] = sum((real.(sol[σ(:e,:g,1)*σ(:g,:e,1)]).+(N_atom_II[l]-1).*real.(sol[σ(:e,:g,1)*σ(:g,:e,2)])).*N_atom_II[l])/n_t
            end

        end

        next!(progress)
    end
end

## Emission linewidth

In [ ]:
function mf_matrix(κ,g_0,Nl,ω_c,ω_0,P,ϕ,γ,sz)

    M = zeros(ComplexF64,2,2)
    Γ = P+γ+4*ϕ
    sz_n = (P-γ)/(P+γ)

    M[1,1]=-im*ω_c-κ/2
    M[1,2]=-im*g_0*Nl

    M[2,1]=im*g_0*sz
    M[2,2]=-im*ω_0-Γ/2

    return M

end

eigs = zeros(ComplexF64,num_O,num_Nl,2)
gap = zeros(num_O,num_Nl)

Threads.@threads for i in 1:num_O
    for j in 1:num_Nl
        eigs[i,j,:] = eigvals(mf_matrix((dec_f[j]+kappa),sqrt(g_02[j]),N_atom_II[j],-delta_lamb_f[j],w_0[j],P[i]+P_eff[j],dep_eff[j],dec_eff[j],2*M[i,j]-1))
        gap[i,j] = -2*real.(eigs[i,j,2])
        if gap[i,j] < 0 || it[i,j]>10
            gap[i,j] = NaN
        end
    end
end

## Save

In [ ]:
data_dir = joinpath("..", "data", "P_Ng") # Relative to this notebook

npzwrite(joinpath(data_dir, "eff_n.npy"), n)
npzwrite(joinpath(data_dir, "eff_linewidth.npy"), gap)
npzwrite(joinpath(data_dir, "eff_SdagS.npy"), corr)
npzwrite(joinpath(data_dir, "eff_pop_gain.npy"), M)